# AgriNav — NASA POWER Crop Risk & Crop Suitability

One notebook, two parts, **no duplicated code between them**:

1. **Training** — fetches real NASA POWER data for all 64 Bangladesh districts (2020–2026), trains the risk model and the crop-suitability model.
2. **Prediction / analysis** — reuses the exact functions and constants defined in part 1 (same notebook namespace, nothing re-imported or re-typed) to answer district/date questions, build a national risk map, generate alerts, compare years, rank districts, and project a full-year seasonal outlook.

Every number that comes out of this notebook traces back to an actual NASA POWER reading — no synthetic or third-party data anywhere.

## Setup (Colab: run this first)

In [ ]:
!pip install -q pandas numpy scikit-learn joblib requests matplotlib


# Part 1 — Training

## Imports & date defaults

In [ ]:
import os
from datetime import datetime, timedelta

import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split

# NASA POWER's daily dataset has a processing lag of a few days, so "today"
# is never actually available yet -- default the end date to 5 days back.
DEFAULT_START = "20200101"
DEFAULT_END = (datetime.today() - timedelta(days=5)).strftime("%Y%m%d")


## Feature/target definitions, crop requirement rules, NASA POWER request config

In [ ]:
RISK_FEATURES = [
    "precipitation",
    "temperature",
    "humidity",
    "solar_radiation",
    "wind_speed",
    "precip_7d",
    "temp_7d_avg",
]
RISK_TARGET = "risk_label"
RISK_CLASSES = ["low", "medium", "high"]

CROP_FEATURES = ["temperature", "humidity", "precip_7d", "temp_7d_avg"]
CROP_TARGET = "best_crop"

# Agro-climatic requirement ranges for common Bangladesh crops.
# (temp_min, temp_max) in Celsius, (rain_min, rain_max) = ideal 7-day
# cumulative rainfall in mm. Source: standard agronomy references (BARI/BRRI
# crop calendars) — used here as rule-based labeling, not a NASA field.
CROP_REQUIREMENTS = {
    "Aman Rice": {"temp": (20, 35), "rain_7d": (30, 120)},
    "Boro Rice": {"temp": (18, 33), "rain_7d": (0, 30)},
    "Wheat": {"temp": (10, 25), "rain_7d": (0, 20)},
    "Maize": {"temp": (18, 32), "rain_7d": (10, 60)},
    "Jute": {"temp": (24, 37), "rain_7d": (40, 150)},
    "Potato": {"temp": (15, 25), "rain_7d": (0, 20)},
    "Mustard": {"temp": (10, 25), "rain_7d": (0, 15)},
}

DATA_DIR = "data"
RAW_CSV = os.path.join(DATA_DIR, "nasa_power_raw.csv")
RISK_CSV = os.path.join(DATA_DIR, "risk_training_data.csv")
CROP_CSV = os.path.join(DATA_DIR, "crop_training_data.csv")

POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
POWER_PARAMS = ["T2M", "PRECTOTCORR", "RH2M", "ALLSKY_SFC_SW_DWN", "WS2M"]
POWER_COLUMN_MAP = {
    "T2M": "temperature",
    "PRECTOTCORR": "precipitation",
    "RH2M": "humidity",
    "ALLSKY_SFC_SW_DWN": "solar_radiation",
    "WS2M": "wind_speed",
}


## All 64 Bangladesh districts (lat/lon)

In [ ]:
# All 64 Bangladesh districts (name, lat, lon) — one point per district so the
# training data covers the country's full climate variation, not just a few
# major cities. Coordinates are each district's headquarters town.
BD_DISTRICTS = [
    # Dhaka division
    ("Dhaka", 23.8103, 90.4125),
    ("Faridpur", 23.6070, 89.8429),
    ("Gazipur", 23.9999, 90.4203),
    ("Gopalganj", 23.0050, 89.8266),
    ("Kishoreganj", 24.4449, 90.7766),
    ("Madaripur", 23.1641, 90.1897),
    ("Manikganj", 23.8644, 90.0047),
    ("Munshiganj", 23.5422, 90.5305),
    ("Narayanganj", 23.6238, 90.5000),
    ("Narsingdi", 23.9322, 90.7150),
    ("Rajbari", 23.7574, 89.6444),
    ("Shariatpur", 23.2423, 90.4348),
    ("Tangail", 24.2513, 89.9167),
    # Mymensingh division
    ("Mymensingh", 24.7471, 90.4203),
    ("Jamalpur", 24.9375, 89.9375),
    ("Netrokona", 24.8709, 90.7276),
    ("Sherpur", 25.0205, 90.0153),
    # Chattogram division
    ("Chattogram", 22.3569, 91.7832),
    ("Cox's Bazar", 21.4272, 92.0058),
    ("Cumilla", 23.4607, 91.1809),
    ("Brahmanbaria", 23.9571, 91.1119),
    ("Chandpur", 23.2333, 90.6667),
    ("Feni", 23.0159, 91.3976),
    ("Khagrachhari", 23.1193, 91.9847),
    ("Lakshmipur", 22.9447, 90.8282),
    ("Noakhali", 22.8696, 91.0995),
    ("Rangamati", 22.7324, 92.1936),
    ("Bandarban", 22.1953, 92.2184),
    # Rajshahi division
    ("Rajshahi", 24.3745, 88.6042),
    ("Bogura", 24.8465, 89.3773),
    ("Joypurhat", 25.0968, 89.0227),
    ("Naogaon", 24.7936, 88.9318),
    ("Natore", 24.4206, 88.9873),
    ("Chapainawabganj", 24.5965, 88.2775),
    ("Pabna", 24.0064, 89.2372),
    ("Sirajganj", 24.4534, 89.7010),
    # Khulna division
    ("Khulna", 22.8456, 89.5403),
    ("Bagerhat", 22.6602, 89.7895),
    ("Chuadanga", 23.6402, 88.8410),
    ("Jashore", 23.1667, 89.2167),
    ("Jhenaidah", 23.5450, 89.1539),
    ("Kushtia", 23.9013, 89.1200),
    ("Magura", 23.4873, 89.4198),
    ("Meherpur", 23.7622, 88.6318),
    ("Narail", 23.1725, 89.5126),
    ("Satkhira", 22.7185, 89.0705),
    # Barishal division
    ("Barishal", 22.7010, 90.3535),
    ("Barguna", 22.1591, 90.1120),
    ("Bhola", 22.6859, 90.6482),
    ("Jhalokati", 22.6406, 90.1987),
    ("Patuakhali", 22.3596, 90.3296),
    ("Pirojpur", 22.5841, 89.9720),
    # Sylhet division
    ("Sylhet", 24.8949, 91.8687),
    ("Habiganj", 24.3745, 91.4155),
    ("Moulvibazar", 24.4829, 91.7774),
    ("Sunamganj", 25.0658, 91.3950),
    # Rangpur division
    ("Rangpur", 25.7439, 89.2752),
    ("Dinajpur", 25.6217, 88.6354),
    ("Gaibandha", 25.3288, 89.5286),
    ("Kurigram", 25.8072, 89.6293),
    ("Lalmonirhat", 25.9923, 89.2847),
    ("Nilphamari", 25.9317, 88.8560),
    ("Panchagarh", 26.3411, 88.5541),
    ("Thakurgaon", 26.0336, 88.4616),
]


## Fetch real data from the NASA POWER API

In [ ]:
def fetch_power_data(lat, lon, start, end):
    """Pulls daily meteorological data for one point from the NASA POWER API.
    No auth needed, it's a public endpoint. Returns a DataFrame indexed by date.
    """
    params = {
        "parameters": ",".join(POWER_PARAMS),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": start,
        "end": end,
        "format": "JSON",
    }
    resp = requests.get(POWER_URL, params=params, timeout=60)
    resp.raise_for_status()
    parameter_data = resp.json()["properties"]["parameter"]

    df = pd.DataFrame(parameter_data)
    df.index = pd.to_datetime(df.index, format="%Y%m%d")
    df.index.name = "date"
    df = df.rename(columns=POWER_COLUMN_MAP)
    df = df.replace(-999, np.nan).dropna()
    return df.sort_index()


## Rule-based labeling (risk level & best-fit crop) -- derived from the real NASA values above, nothing invented

In [ ]:
def label_risk_row(row):
    """Domain-threshold labeling: high risk if the land is drying out
    (low 7-day rainfall) or under sustained heat stress; medium risk on
    borderline values; low risk otherwise."""
    if row["precip_7d"] < 5 or row["temp_7d_avg"] > 35:
        return 2  # high
    if row["precip_7d"] < 20 or row["temp_7d_avg"] > 32:
        return 1  # medium
    return 0  # low


def crop_fit_score(row, requirements):
    """Higher is better: 0 when temp/rain sit at the range center, more
    negative the further outside the crop's ideal range they fall."""
    t_lo, t_hi = requirements["temp"]
    r_lo, r_hi = requirements["rain_7d"]
    t_mid, t_half = (t_lo + t_hi) / 2, (t_hi - t_lo) / 2
    r_mid, r_half = (r_lo + r_hi) / 2, (r_hi - r_lo) / 2

    t_dev = (row["temp_7d_avg"] - t_mid) / t_half
    r_dev = (row["precip_7d"] - r_mid) / r_half
    return -(t_dev**2 + r_dev**2)


def label_best_crop(row):
    scores = {crop: crop_fit_score(row, reqs) for crop, reqs in CROP_REQUIREMENTS.items()}
    return max(scores, key=scores.get)


## Build the training datasets (fetches + incrementally caches to `data/`)

In [ ]:
def build_power_dataset(locations=BD_DISTRICTS, start=DEFAULT_START, end=DEFAULT_END, refresh=False):
    """Fetches real NASA POWER daily data for each location and engineers
    the rolling features shared by both models. Saved incrementally to
    data/nasa_power_raw.csv — one district at a time, flushed to disk and
    printed immediately, so progress is visible on disk even mid-run.
    Re-running resumes from whichever districts are already in the CSV;
    pass refresh=True to wipe the cache and re-fetch everything."""
    os.makedirs(DATA_DIR, exist_ok=True)
    if refresh and os.path.exists(RAW_CSV):
        os.remove(RAW_CSV)

    done = set()
    if os.path.exists(RAW_CSV):
        done = set(pd.read_csv(RAW_CSV, usecols=["location"])["location"].unique())
        print(f"Resuming: {len(done)}/{len(locations)} districts already in {RAW_CSV}", flush=True)

    remaining = [loc for loc in locations if loc[0] not in done]
    for name, lat, lon in remaining:
        print(f"Fetching NASA POWER data for {name} ({lat}, {lon})...", flush=True)
        df = fetch_power_data(lat, lon, start, end)
        df["precip_7d"] = df["precipitation"].rolling(7, min_periods=1).sum()
        df["temp_7d_avg"] = df["temperature"].rolling(7, min_periods=1).mean()
        df["location"] = name
        df = df.reset_index()
        df.to_csv(RAW_CSV, mode="a", header=not os.path.exists(RAW_CSV), index=False)
        print(f"  -> saved {len(df)} rows for {name} ({len(done) + 1}/{len(locations)})", flush=True)
        done.add(name)

    full = pd.read_csv(RAW_CSV, parse_dates=["date"])
    print(
        f"NASA POWER raw dataset complete: {len(full)} rows, "
        f"{full['location'].nunique()} districts -> {RAW_CSV}",
        flush=True,
    )
    return full


def build_power_risk_dataset(locations=BD_DISTRICTS, start=DEFAULT_START, end=DEFAULT_END, refresh=False):
    full = build_power_dataset(locations, start, end, refresh=refresh)
    full[RISK_TARGET] = full.apply(label_risk_row, axis=1)
    os.makedirs(DATA_DIR, exist_ok=True)
    full.to_csv(RISK_CSV, index=False)
    print(f"Saved risk training data -> {RISK_CSV}")
    return full


def build_power_crop_dataset(locations=BD_DISTRICTS, start=DEFAULT_START, end=DEFAULT_END, refresh=False):
    full = build_power_dataset(locations, start, end, refresh=refresh)
    full[CROP_TARGET] = full.apply(label_best_crop, axis=1)
    os.makedirs(DATA_DIR, exist_ok=True)
    full.to_csv(CROP_CSV, index=False)
    print(f"Saved crop training data -> {CROP_CSV}")
    return full


## Chronological train/test split + training functions

In [ ]:
def chronological_split(df: pd.DataFrame, test_frac=0.2):
    """Splits by date instead of randomly. A random row split would leak
    information between adjacent days at the same location (precip_7d and
    temp_7d_avg are rolling windows, so neighboring rows are correlated) and
    make accuracy look better than real-world generalization. Holding out the
    most recent slice of dates for every location is the honest test: the
    model has never seen those dates for any location during training."""
    dates = np.sort(df["date"].unique())
    cutoff = dates[int(len(dates) * (1 - test_frac))]
    train_df = df[df["date"] < cutoff]
    test_df = df[df["date"] >= cutoff]
    print(f"Train dates: {train_df['date'].min().date()} -> {train_df['date'].max().date()}")
    print(f"Test dates:  {test_df['date'].min().date()} -> {test_df['date'].max().date()}")
    return train_df, test_df


def train_risk_model(df: pd.DataFrame, out_path: str):
    train_df, test_df = chronological_split(df)
    X_train, y_train = train_df[RISK_FEATURES], train_df[RISK_TARGET]
    X_test, y_test = test_df[RISK_FEATURES], test_df[RISK_TARGET]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
    )
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print("\n=== Risk model (NASA POWER) ===")
    print(f"5-fold CV accuracy on training data: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
    print("Held-out future dates:")
    print(classification_report(y_test, y_pred, target_names=RISK_CLASSES))
    print(confusion_matrix(y_test, y_pred))
    print(
        pd.Series(model.feature_importances_, index=RISK_FEATURES)
        .sort_values(ascending=False)
        .rename("importance")
    )

    joblib.dump(model, out_path)
    print(f"Saved risk model -> {out_path}")


def train_crop_model(df: pd.DataFrame, out_path: str):
    train_df, test_df = chronological_split(df)
    X_train, y_train = train_df[CROP_FEATURES], train_df[CROP_TARGET]
    X_test, y_test = test_df[CROP_FEATURES], test_df[CROP_TARGET]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        class_weight="balanced",
    )
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    print("\n=== Crop suitability model (NASA POWER) ===")
    print(f"5-fold CV accuracy on training data: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")
    print("Held-out future dates:")
    print(classification_report(y_test, y_pred))
    print(
        pd.Series(model.feature_importances_, index=CROP_FEATURES)
        .sort_values(ascending=False)
        .rename("importance")
    )

    joblib.dump(model, out_path)
    print(f"Saved crop model -> {out_path}")


## Train the risk model
First run fetches live data for all 64 districts (2020 -> today-5d) and caches it to `data/nasa_power_raw.csv`; later runs resume/reuse the cache (pass `refresh=True` to force a full re-fetch).

In [ ]:
risk_df = build_power_risk_dataset()
train_risk_model(risk_df, "crop_risk_model.pkl")


## Train the crop-suitability model

In [ ]:
crop_df = build_power_crop_dataset()
train_crop_model(crop_df, "crop_suitability_model.pkl")


# Part 2 — Prediction & analysis

Everything below reuses `BD_DISTRICTS`, `RISK_FEATURES`, `CROP_FEATURES`, `CROP_REQUIREMENTS`, `fetch_power_data`, `crop_fit_score`, `RAW_CSV`, `DATA_DIR`, etc. from Part 1 directly -- no re-imports, no re-typed constants.

## Model paths + district lookup

In [ ]:
RISK_MODEL_PATH = "crop_risk_model.pkl"
CROP_MODEL_PATH = "crop_suitability_model.pkl"

DISTRICT_COORDS = {name: (lat, lon) for name, lat, lon in BD_DISTRICTS}


def load_models():
    if not (os.path.exists(RISK_MODEL_PATH) and os.path.exists(CROP_MODEL_PATH)):
        raise FileNotFoundError("Model files not found. Run the training cells above first.")
    return joblib.load(RISK_MODEL_PATH), joblib.load(CROP_MODEL_PATH)


## Feature lookup: actual NASA reading (cached or live) or NASA climatology

In [ ]:
def get_features_for(district, date_str, cache=None):
    """Returns a dict of the 7 raw+rolling features for one district on one
    date. Uses the cached NASA POWER dataset when the date is already in it;
    otherwise fetches the last 7 days from NASA POWER live."""
    if district not in DISTRICT_COORDS:
        raise ValueError(f"Unknown district '{district}'. See BD_DISTRICTS.")

    target_date = pd.to_datetime(date_str, format="%Y%m%d")

    if cache is None and os.path.exists(RAW_CSV):
        cache = pd.read_csv(RAW_CSV, parse_dates=["date"])

    if cache is not None:
        row = cache[(cache["location"] == district) & (cache["date"] == target_date)]
        if not row.empty:
            return row.iloc[0].to_dict()

    lat, lon = DISTRICT_COORDS[district]
    start = (target_date - pd.Timedelta(days=7)).strftime("%Y%m%d")
    end = target_date.strftime("%Y%m%d")
    df = fetch_power_data(lat, lon, start, end)
    if target_date not in df.index:
        raise ValueError(
            f"NASA POWER has no data for {district} on {target_date.date()} "
            "(too recent -- POWER has a few days' processing lag, or too far in the future)."
        )
    df["precip_7d"] = df["precipitation"].rolling(7, min_periods=1).sum()
    df["temp_7d_avg"] = df["temperature"].rolling(7, min_periods=1).mean()
    return df.loc[target_date].to_dict()


def climatology_features(cache, district, target_date, window_days=10):
    """Average of the REAL NASA POWER values recorded on this same calendar
    day (+/- window_days) across every year already in the cache. This is
    what lets `climatology_outlook`/`year_outlook` answer questions about
    dates NASA hasn't measured yet, without inventing any data or bolting on
    a separate forecasting model -- the standard climatology-as-forecast
    technique. Every value averaged is a genuine NASA POWER reading; the
    'source_years' field returned makes that traceable."""
    sub = cache[cache["location"] == district].copy()
    if sub.empty:
        raise ValueError(f"No cached NASA POWER data for '{district}'.")

    target_doy = target_date.dayofyear
    doy = sub["date"].dt.dayofyear
    diff = (doy - target_doy).abs()
    diff = np.minimum(diff, 365 - diff)  # wrap around the Dec/Jan boundary
    match = sub[diff <= window_days]
    if match.empty:
        raise ValueError(f"Not enough historical NASA data near day-of-year {target_doy} for '{district}'.")

    cols = ["precipitation", "temperature", "humidity", "solar_radiation", "wind_speed", "precip_7d", "temp_7d_avg"]
    features = match[cols].mean().to_dict()
    years = sorted(match["date"].dt.year.unique().tolist())
    features["years_averaged"] = len(years)
    features["years_list"] = ",".join(str(y) for y in years)
    features["samples_averaged"] = int(len(match))
    return features


def get_features_any(district, date_str, cache):
    """Actual NASA reading if available; falls back to NASA POWER
    climatology for dates too far in the future."""
    try:
        return get_features_for(district, date_str, cache=cache), "actual"
    except ValueError:
        target_date = pd.to_datetime(date_str, format="%Y%m%d")
        return climatology_features(cache, district, target_date), "climatology"


def score_crop_fit(features, crop):
    return crop_fit_score(features, CROP_REQUIREMENTS[crop])


## Core prediction functions

In [ ]:
def predict_one(district, date_str, risk_model, crop_model, cache=None):
    features = get_features_for(district, date_str, cache=cache)
    risk_x = pd.DataFrame([{k: features[k] for k in RISK_FEATURES}])
    crop_x = pd.DataFrame([{k: features[k] for k in CROP_FEATURES}])

    risk_idx = risk_model.predict(risk_x)[0]
    risk_proba = risk_model.predict_proba(risk_x)[0]
    best_crop = crop_model.predict(crop_x)[0]

    return {
        "district": district,
        "date": date_str,
        "risk": RISK_CLASSES[risk_idx],
        "risk_confidence": round(float(risk_proba.max()), 3),
        "best_crop": best_crop,
        "precipitation": round(float(features["precipitation"]), 2),
        "temperature": round(float(features["temperature"]), 2),
        "precip_7d": round(float(features["precip_7d"]), 2),
        "temp_7d_avg": round(float(features["temp_7d_avg"]), 2),
    }


def outlook_one(district, target_date, cache, risk_model, crop_model):
    features = climatology_features(cache, district, target_date)
    risk_x = pd.DataFrame([{k: features[k] for k in RISK_FEATURES}])
    crop_x = pd.DataFrame([{k: features[k] for k in CROP_FEATURES}])

    risk_idx = risk_model.predict(risk_x)[0]
    risk_proba = risk_model.predict_proba(risk_x)[0]
    best_crop = crop_model.predict(crop_x)[0]

    return {
        "date": target_date.date(),
        "risk": RISK_CLASSES[risk_idx],
        "risk_confidence": round(float(risk_proba.max()), 3),
        "best_crop": best_crop,
        "avg_precip_7d": round(float(features["precip_7d"]), 2),
        "avg_temp_7d_avg": round(float(features["temp_7d_avg"]), 2),
        "years_averaged": features["years_averaged"],
        "source_years": features["years_list"],
    }


## 1. Predict — one district, one past/present date

In [ ]:
def predict_district(district, date):
    risk_model, crop_model = load_models()
    result = predict_one(district, date, risk_model, crop_model)
    for k, v in result.items():
        print(f"{k:16s}: {v}")
    return result


In [ ]:
predict_district("Dhaka", "20260601")


## 2. National snapshot — all 64 districts, one date

In [ ]:
def national_snapshot(date):
    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"]) if os.path.exists(RAW_CSV) else None

    rows = []
    for district in DISTRICT_COORDS:
        try:
            rows.append(predict_one(district, date, risk_model, crop_model, cache=cache))
        except ValueError as e:
            print(f"Skipping {district}: {e}")

    snapshot = pd.DataFrame(rows).sort_values("risk", ascending=False)
    os.makedirs(DATA_DIR, exist_ok=True)
    out_path = os.path.join(DATA_DIR, f"national_risk_snapshot_{date}.csv")
    snapshot.to_csv(out_path, index=False)

    print(snapshot.to_string(index=False))
    print(f"\nHigh-risk districts: {(snapshot['risk'] == 'high').sum()} / {len(snapshot)}")
    print(f"Saved -> {out_path}")
    return snapshot


In [ ]:
national_snapshot("20260601")


## 3. Crop calendar — most-recommended crop per district per month

In [ ]:
def crop_calendar_report():
    if not os.path.exists(CROP_CSV):
        raise FileNotFoundError(f"{CROP_CSV} not found. Run the crop-model training cell first.")

    df = pd.read_csv(CROP_CSV, parse_dates=["date"])
    df["month"] = df["date"].dt.strftime("%b")

    calendar = (
        df.groupby(["location", "month"])[CROP_TARGET]
        .agg(lambda s: s.mode().iloc[0])
        .unstack("month")
    )
    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    calendar = calendar[[m for m in month_order if m in calendar.columns]]

    out_path = os.path.join(DATA_DIR, "crop_calendar.csv")
    calendar.to_csv(out_path)
    print(calendar.to_string())
    print(f"\nSaved -> {out_path}")
    return calendar


In [ ]:
crop_calendar_report()


## 4. Climatology outlook — risk + crop for a future date, one district

In [ ]:
def climatology_outlook(district, date):
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"{RAW_CSV} not found. Run the training cells first.")
    if district not in DISTRICT_COORDS:
        raise ValueError(f"Unknown district '{district}'. See BD_DISTRICTS.")

    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"])
    target_date = pd.to_datetime(date, format="%Y%m%d")

    result = outlook_one(district, target_date, cache, risk_model, crop_model)
    print(f"Climatology outlook for {district} around {result['date']}")
    print(f"(Average of REAL NASA POWER readings from {result['source_years']} "
          f"on this same calendar day +/- 10 days -- no invented values.)")
    for k, v in result.items():
        print(f"{k:16s}: {v}")
    return result


In [ ]:
climatology_outlook("Dhaka", "20271215")


## 5. Year-round seasonal outlook — when rain, when drought, for one district

In [ ]:
def year_outlook(district, year):
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"{RAW_CSV} not found. Run the training cells first.")
    if district not in DISTRICT_COORDS:
        raise ValueError(f"Unknown district '{district}'. See BD_DISTRICTS.")

    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"])
    year = int(year)

    rows = []
    for month in range(1, 13):
        target_date = pd.Timestamp(year=year, month=month, day=15)
        result = outlook_one(district, target_date, cache, risk_model, crop_model)
        result["month"] = target_date.strftime("%B")
        rows.append(result)

    table = pd.DataFrame(rows)[["month", "avg_precip_7d", "avg_temp_7d_avg", "risk", "best_crop", "source_years"]]

    note = {
        "low": "normal",
        "medium": "moderate risk — keep watch",
        "high": "\u26A0\ufe0f drought/heat risk",
    }
    table["note"] = table["risk"].map(note)

    print(f"Year-round climatology outlook for {district}, {year}")
    print("Every number below is a real-NASA-data average -- see 'source_years'.")
    print(table.to_string(index=False))

    os.makedirs(DATA_DIR, exist_ok=True)
    out_path = os.path.join(DATA_DIR, f"season_{district}_{year}.csv")
    table.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    return table


In [ ]:
year_outlook("Rajshahi", "2027")


## 6. Risk map — PNG of all 64 districts colored by risk

In [ ]:
def risk_map(date):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"]) if os.path.exists(RAW_CSV) else None

    rows = []
    for district in DISTRICT_COORDS:
        try:
            rows.append(predict_one(district, date, risk_model, crop_model, cache=cache))
        except ValueError as e:
            print(f"Skipping {district}: {e}")

    snapshot = pd.DataFrame(rows)
    color_map = {"low": "#2ca02c", "medium": "#f0ad4e", "high": "#d62728"}
    colors = snapshot["risk"].map(color_map)
    lats = [DISTRICT_COORDS[d][0] for d in snapshot["district"]]
    lons = [DISTRICT_COORDS[d][1] for d in snapshot["district"]]

    fig, ax = plt.subplots(figsize=(7, 9))
    ax.scatter(lons, lats, c=colors, s=110, edgecolors="black", linewidths=0.5, zorder=3)
    for lon, lat, name in zip(lons, lats, snapshot["district"]):
        ax.annotate(name, (lon, lat), fontsize=5.5, ha="center", va="bottom", xytext=(0, 4), textcoords="offset points")
    ax.set_title(f"AgriNav -- drought/heat risk by district ({date})\nNASA POWER data")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=10, label=lbl)
               for lbl, c in color_map.items()]
    ax.legend(handles=handles, title="Risk", loc="lower right")
    fig.tight_layout()

    os.makedirs(DATA_DIR, exist_ok=True)
    out_path = os.path.join(DATA_DIR, f"risk_map_{date}.png")
    fig.savefig(out_path, dpi=150)
    plt.show()
    print(f"Saved -> {out_path}")
    print(snapshot["risk"].value_counts().to_string())
    return out_path


In [ ]:
risk_map("20260601")


## 7. Alerts — plain-language high-risk warnings, all districts

In [ ]:
def risk_alerts(date):
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"{RAW_CSV} not found. Run the training cells first.")

    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"])
    target_date = pd.to_datetime(date, format="%Y%m%d")

    alerts = []
    for district in DISTRICT_COORDS:
        try:
            features, source = get_features_any(district, date, cache)
        except ValueError as e:
            print(f"Skipping {district}: {e}")
            continue

        risk_x = pd.DataFrame([{k: features[k] for k in RISK_FEATURES}])
        crop_x = pd.DataFrame([{k: features[k] for k in CROP_FEATURES}])
        risk_idx = risk_model.predict(risk_x)[0]
        risk_label = RISK_CLASSES[risk_idx]
        best_crop = crop_model.predict(crop_x)[0]

        if risk_label == "high":
            reason = "low rainfall / drought risk" if features["precip_7d"] < 5 else "excess heat"
            provenance = (
                f"NASA POWER real years averaged: {features['years_list']}"
                if source == "climatology"
                else f"NASA POWER actual reading for {target_date.date()}"
            )
            alerts.append(
                f"\U0001F6A8 {district} -- {target_date.date()} ({source}): HIGH risk ({reason}). "
                f"Recommended crop: {best_crop}. avg_precip_7d={features['precip_7d']:.1f}mm, "
                f"avg_temp_7d={features['temp_7d_avg']:.1f}C  [{provenance}]"
            )

    os.makedirs(DATA_DIR, exist_ok=True)
    out_path = os.path.join(DATA_DIR, f"alerts_{date}.txt")
    with open(out_path, "w") as f:
        f.write("\n".join(alerts) if alerts else "No high-risk districts for this date.\n")

    print(f"{len(alerts)} high-risk district(s) out of {len(DISTRICT_COORDS)}")
    for a in alerts:
        print(a)
    print(f"\nSaved -> {out_path}")
    return alerts


In [ ]:
risk_alerts("20271215")


## 8. Compare — one district's actual NASA data across real years

In [ ]:
def compare_years(district, years, month_day):
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"{RAW_CSV} not found. Run the training cells first.")
    if district not in DISTRICT_COORDS:
        raise ValueError(f"Unknown district '{district}'. See BD_DISTRICTS.")

    risk_model, crop_model = load_models()
    cache = pd.read_csv(RAW_CSV, parse_dates=["date"])

    rows = []
    for year in years:
        date_str = f"{year}{month_day}"
        try:
            result = predict_one(district, date_str, risk_model, crop_model, cache=cache)
        except ValueError as e:
            print(f"Skipping {year}: {e}")
            continue
        rows.append(result)

    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    out_path = os.path.join(DATA_DIR, f"compare_{district}_{month_day}.csv")
    table.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    return table


In [ ]:
compare_years("Rangpur", ["2021", "2022", "2023", "2024", "2025"], "0715")


## 9. Rank — all 64 districts, suitability for one crop

In [ ]:
def rank_districts(crop, date):
    if crop not in CROP_REQUIREMENTS:
        raise ValueError(f"Unknown crop '{crop}'. Choices: {list(CROP_REQUIREMENTS)}")
    if not os.path.exists(RAW_CSV):
        raise FileNotFoundError(f"{RAW_CSV} not found. Run the training cells first.")

    cache = pd.read_csv(RAW_CSV, parse_dates=["date"])

    rows = []
    for district in DISTRICT_COORDS:
        try:
            features, source = get_features_any(district, date, cache)
        except ValueError as e:
            print(f"Skipping {district}: {e}")
            continue
        rows.append({
            "district": district,
            "fit_score": round(score_crop_fit(features, crop), 3),
            "source": source,
            "precip_7d": round(float(features["precip_7d"]), 2),
            "temp_7d_avg": round(float(features["temp_7d_avg"]), 2),
        })

    table = pd.DataFrame(rows).sort_values("fit_score", ascending=False)
    print(f"District ranking for {crop} on {date} (0 = ideal, more negative = worse fit)")
    print(table.to_string(index=False))
    out_path = os.path.join(DATA_DIR, f"rank_{crop.replace(' ', '_')}_{date}.csv")
    table.to_csv(out_path, index=False)
    print(f"\nSaved -> {out_path}")
    return table


In [ ]:
rank_districts("Aman Rice", "20260701")
